# 18tobias_footprint

Plot TF footprints from TOBIAS BINDetect `bed_signal.txt` outputs.

Extracted from `18tobias_filtered_JASPR.ipynb`:
- **Cell A** (was cell #21): per region × NT — raw footprint curves + CUSUS − CON diff curves.
- **Cell B** (was cell #144): per region × TF — multi-celltype line plots with motif boundaries.

The notebook is standalone: it does **not** depend on any variable defined in the
parent notebook. Users provide either:
- a `focus_motif_results` DataFrame (Cell A), or
- a `ct_mmotif` DataFrame (Cell B).

Required columns are documented in the corresponding entry cells below.

## 1. Imports

In [ ]:
import os
import math
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.lines import Line2D

sns.set_theme(style='whitegrid')

## 2. Configuration

In [ ]:
# Root directory produced by the TOBIAS pipeline (contains per-region BINDetect outputs).
TOBIASBAM_DIR = '/data2st2/junyi/output/atac1112/tobiasbam'

# Where to write figures / summary tables.
OUTPUT_DIR = '/home/junyichen/code/scmmd/figures/tobias_footprints'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cell A scope.
REGIONS_TO_PLOT = ['AMY', 'PFC', 'HIP']
NTS_TO_PLOT     = ['GABA', 'Glut', 'NN']

# Cell B scope (note: original cell #144 used 'HPF' instead of 'HIP').
REGIONS_CELL_B  = ['PFC', 'AMY', 'HPF']

# Motif boundaries (TOBIAS default ±50bp around centre).
MOTIF_BOUNDARIES = [54, 59, 64]

# Group colours.
COLOR_MAP = {'MC': 'tab:orange', 'MW': 'tab:blue'}

# Bed-signal path template. Cell A uses the Jaspar26 results; Cell B uses cisbp_BIND.
BED_SIGNAL_PATH_CELL_A = (
    '{tobiasbam}/{region}_Jaspar26/{celltype}_MC_footprints.bw/{motif}/beds/{motif}_all.bed_signal.txt'
)
BED_SIGNAL_PATH_CELL_B = (
    '{tobiasbam}/{region2}cisbp_BIND/{celltype}_footprints.bw/{motif}/beds/{motif}_all.bed_signal.txt'
)

## 3. Utilities

In [ ]:
def read_bed_signal(fname):
    """Read a TOBIAS `bed_signal.txt` file into a (rows × positions) DataFrame.

    The file format is:
        # header lines ...
        <peak_id>\t<other_col>\t<pos0,pos1,...,posN>

    Returns
    -------
    pd.DataFrame
        Index is the peak identifier (column 0); columns are integer positions.
    """
    df_tmp = pd.read_csv(fname, sep='\t', header=None, skiprows=2)
    df_signal = df_tmp[2].str.split(',', expand=True).astype(float)
    df_signal.index = df_tmp[0]
    return df_signal


def get_nt_label(celltype_name):
    """Bucket a celltype name into GABA / Glut / NN (non-neuronal)."""
    if 'GABA' in celltype_name:
        return 'GABA'
    if 'Glut' in celltype_name:
        return 'Glut'
    return 'NN'


def region_from_celltype(celltype):
    """Extract the region prefix from a celltype name (e.g. 'AMY_CA1_Glut_MC' -> 'AMY')."""
    return celltype.split('_')[0]

## 4. Entry A — per region × NT raw + diff curves (was cell #21)

Provide `focus_motif_results` with at least these columns:
- `ctname` — celltype name **including the `_corrected` suffix** (matches the folder name under `tobiasbam/<region>_Jaspar26/`)
- `output_prefix` — TOBIAS motif directory name (the folder under `<ct>_footprints.bw/<motif>/`)
- `TF` — TF label used for plotting / file naming

In [ ]:
# Provide your own DataFrame here. Example skeleton:
#
# focus_motif_results = pd.read_csv('/path/to/focus_motif_results_Jaspar26.csv')
#
# Or build it from BINDetect outputs (see original cell #14-#17 of
# 18tobias_filtered_JASPR.ipynb for `add_columns` / motif filtering).

focus_motif_results = None  # <- replace with a real DataFrame
assert focus_motif_results is not None, (
    'Set `focus_motif_results` to a DataFrame with columns: '
    'ctname, output_prefix, TF (plus any extras you need).'
)

In [ ]:
output_dir = OUTPUT_DIR
regions_to_plot = REGIONS_TO_PLOT
nts_to_plot = NTS_TO_PLOT
all_diff_summaries = []

for region_name in regions_to_plot:
    for nt_name in nts_to_plot:
        df_signal_all = pd.DataFrame()

        for _, row in focus_motif_results.iterrows():
            celltype = str(row['ctname']).replace('_corrected', '')
            if celltype.split('_')[0] != region_name:
                continue
            if get_nt_label(celltype) != nt_name:
                continue

            motif = row['output_prefix']
            tf_name = row['TF']
            fname = BED_SIGNAL_PATH_CELL_A.format(
                tobiasbam=TOBIASBAM_DIR,
                region=region_name,
                celltype=celltype,
                motif=motif,
            )
            if not os.path.exists(fname):
                continue

            df_tmp = pd.read_csv(fname, sep='\t', header=None, skiprows=2)
            df_signal = df_tmp[2].str.split(',', expand=True).astype(float)
            df_signal.index = df_tmp[0] + '_' + tf_name
            df_signal_all = pd.concat([df_signal_all, df_signal], axis=0)

        if df_signal_all.empty:
            print(f'Skip {region_name} {nt_name}: no footprint signal found.')
            continue

        plot_df = df_signal_all.copy()
        plot_df.index = plot_df.index.astype(str)
        plot_df = plot_df.reset_index(names='signal_id').melt(
            id_vars='signal_id',
            var_name='position',
            value_name='signal'
        )
        plot_df['TF'] = plot_df['signal_id'].str.rsplit('_', n=1).str[-1]
        plot_df['MC_MW'] = plot_df['signal_id'].str.split('_').str[-3]
        plot_df['position'] = pd.to_numeric(plot_df['position'], errors='coerce')
        plot_df = plot_df[plot_df['MC_MW'].isin(['MC', 'MW'])].dropna(subset=['position', 'signal'])
        plot_df['group'] = plot_df['MC_MW'].map({'MC': 'CUSUS', 'MW': 'CON'})

        if plot_df.empty:
            print(f'Skip {region_name} {nt_name}: no MC/CON rows available after parsing.')
            continue

        safe_prefix = f'{region_name}_{nt_name}'

        g = sns.relplot(
            data=plot_df,
            x='position',
            y='signal',
            hue='group',
            kind='line',
            col='TF',
            col_wrap=4,
            height=3,
            aspect=1.4,
            estimator='mean',
            errorbar='sd',
            palette={'CUSUS': 'tab:orange', 'CON': 'tab:blue'}
        )
        g.fig.suptitle(f'{region_name} {nt_name}: CUSUS vs CON footprint curves', y=1.02)
        g.fig.savefig(os.path.join(output_dir, f'{safe_prefix}_raw_curves.png'), dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(g.fig)

        summary_df = (
            plot_df
            .groupby(['TF', 'position', 'group'], as_index=False)['signal']
            .mean()
        )

        diff_df = (
            summary_df
            .pivot(index=['TF', 'position'], columns='group', values='signal')
            .reset_index()
        )
        diff_df.columns.name = None
        diff_df = diff_df.dropna(subset=['CUSUS', 'CON'])
        if diff_df.empty:
            print(f'Skip {region_name} {nt_name}: unable to compute CUSUS-CON difference.')
            continue

        diff_df['diff_CUSUS_minus_CON'] = diff_df['CUSUS'] - diff_df['CON']
        diff_df['abs_diff'] = diff_df['diff_CUSUS_minus_CON'].abs()

        n_tf = diff_df['TF'].nunique()
        ncols = min(4, n_tf)
        nrows = int(np.ceil(n_tf / ncols))
        fig, axes = plt.subplots(
            nrows=nrows,
            ncols=ncols,
            figsize=(4.5 * ncols, 3 * nrows),
            squeeze=False,
            sharex=True
        )
        axes = axes.flatten()

        for ax, (tf_name, df_tf) in zip(axes, diff_df.groupby('TF')):
            ax.plot(df_tf['position'], df_tf['diff_CUSUS_minus_CON'], color='black', linewidth=1.5)
            ax.fill_between(
                df_tf['position'],
                0,
                df_tf['diff_CUSUS_minus_CON'],
                where=df_tf['diff_CUSUS_minus_CON'] >= 0,
                color='tab:orange',
                alpha=0.3,
                label='CUSUS > CON'
            )
            ax.fill_between(
                df_tf['position'],
                0,
                df_tf['diff_CUSUS_minus_CON'],
                where=df_tf['diff_CUSUS_minus_CON'] < 0,
                color='tab:blue',
                alpha=0.3,
                label='CON > CUSUS'
            )
            ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
            ax.set_title(tf_name)
            ax.set_ylabel('CUSUS - CON')

        for ax in axes[n_tf:]:
            ax.axis('off')

        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles[:2], labels[:2], loc='upper right', frameon=False)
        fig.suptitle(f'{region_name} {nt_name}: difference footprint (CUSUS - CON)', y=1.02)
        fig.savefig(os.path.join(output_dir, f'{safe_prefix}_difference_curves.png'), dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(fig)

        diff_summary = (
            diff_df.groupby('TF', as_index=False)
            .agg(
                mean_abs_diff=('abs_diff', 'mean'),
                max_abs_diff=('abs_diff', 'max'),
                mean_diff=('diff_CUSUS_minus_CON', 'mean')
            )
            .sort_values('mean_abs_diff', ascending=False)
        )
        diff_summary['Region'] = region_name
        diff_summary['NT'] = nt_name
        all_diff_summaries.append(diff_summary)
        diff_summary.to_csv(os.path.join(output_dir, f'{safe_prefix}_diff_summary.csv'), index=False)

if all_diff_summaries:
    diff_summary_all = pd.concat(all_diff_summaries, axis=0, ignore_index=True)
    diff_summary_all.to_csv(os.path.join(output_dir, 'all_region_nt_diff_summary.csv'), index=False)
else:
    diff_summary_all = pd.DataFrame()

print(f'All figures and summaries were saved to: {output_dir}')
diff_summary_all

## 5. Entry B — per region × TF multi-celltype plot (was cell #144)

Provide `ct_mmotif` with at least these columns:
- `celltype` — celltype name; the region prefix is derived via `celltype.split('_')[0]`
- `motif` — TOBIAS motif directory name (must match the folder name under `<ct>_footprints.bw/<motif>/`)
- `TF` — TF label
- `Region` — used for filtering (e.g. `'AMY'`, `'PFC'`, `'HPF'`)

In [ ]:
# Provide your own DataFrame here. Example:
#
# ct_mmotif = pd.read_csv('/path/to/ct_mmotif.csv')
#
# Original notebook built this from `df_merge.drop_duplicates(['celltype','motif','TF','Region'])`.

ct_mmotif = None  # <- replace with a real DataFrame
assert ct_mmotif is not None, (
    'Set `ct_mmotif` to a DataFrame with columns: celltype, motif, TF, Region.'
)

In [ ]:
for region in REGIONS_CELL_B:
    for tf in ct_mmotif['TF'].unique():

        df_tf = ct_mmotif[
            (ct_mmotif['TF'] == tf) &
            (ct_mmotif['Region'] == region)
        ]

        if df_tf.shape[0] == 0:
            continue

        # Only keep celltypes whose name does not contain 'MW' (i.e. keep MC rows).
        df_tf = df_tf[~df_tf['celltype'].str.contains('MW')]
        if df_tf.shape[0] == 0:
            continue

        n_plots = df_tf.shape[0]
        ncols = 6
        nrows = math.ceil(n_plots / ncols)

        fig, axes = plt.subplots(
            nrows=nrows,
            ncols=ncols,
            figsize=(ncols * 2.5, nrows * 2),
            squeeze=False
        )
        axes = axes.flatten()

        last_i = -1
        for i, (_, row) in enumerate(df_tf.iterrows()):
            ax = axes[i]
            last_i = i

            celltype = row['celltype'].replace('_corrected', '')
            motif = row['motif']
            TF = row['TF']
            region2 = celltype.split('_')[0]

            fname = BED_SIGNAL_PATH_CELL_B.format(
                tobiasbam=TOBIASBAM_DIR,
                region2=region2,
                celltype=celltype,
                motif=motif,
            )

            if not os.path.exists(fname):
                ax.set_title(celltype.replace('_MC', ''), fontsize=8)
                ax.axis('off')
                continue

            df_tmp = pd.read_csv(fname, sep='\t', header=None, skiprows=2)
            df_signal = df_tmp[2].str.split(',', expand=True).astype(float)
            df_signal.index = df_tmp[0] + '_' + TF

            x = df_signal.columns

            for idx, r in df_signal.iterrows():
                label = idx.split('_')[-3] if '_' in idx else idx
                ax.plot(x, r.values, color=COLOR_MAP.get(label, 'grey'), linewidth=1, alpha=0.7)
            for v in MOTIF_BOUNDARIES:
                ax.axvline(v, color='grey', linestyle='--', linewidth=0.8)

            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(celltype.replace('_MC', ''), fontsize=8)

        # Switch off any unused subplots.
        for j in range(last_i + 1, len(axes)):
            axes[j].axis('off')

        fig.suptitle(f'{tf} footprints ({region})', fontsize=14)

        legend_elements = [
            Line2D([0], [0], color='tab:orange', lw=2, label='CUMS'),
            Line2D([0], [0], color='tab:blue', lw=2, label='Control')
        ]
        fig.legend(
            handles=legend_elements,
            loc='upper right',
            fontsize=10,
            frameon=False
        )

        outdir = os.path.join(OUTPUT_DIR, region, tf)
        os.makedirs(outdir, exist_ok=True)
        outfile = os.path.join(outdir, f'{tf}_footprints_all_celltypes.pdf')
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        fig.savefig(outfile, dpi=300)
        plt.close(fig)